In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RAW_DATA_DIR = Path("../data/raw/parrot_images")
PROCESSED_DATA_DIR = Path("../data/processed")

MIN_IMAGES_PER_CLASS = 5
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

RAW_DATA_DIR.exists()

True

In [2]:
class_records = []

for class_dir in sorted(RAW_DATA_DIR.iterdir()):
    if not class_dir.is_dir():
        continue

    image_paths = [
        image_path
        for image_path in class_dir.iterdir()
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    class_records.append({
        "class_name": class_dir.name,
        "image_count": len(image_paths),
        "image_paths": image_paths,
    })

class_df = pd.DataFrame(class_records)

eligible_class_df = class_df[class_df["image_count"] >= MIN_IMAGES_PER_CLASS].copy()
excluded_class_df = class_df[class_df["image_count"] < MIN_IMAGES_PER_CLASS].copy()

print(f"Total classes: {len(class_df)}")
print(f"Eligible classes: {len(eligible_class_df)}")
print(f"Excluded classes: {len(excluded_class_df)}")
print(f"Images kept: {eligible_class_df['image_count'].sum()}")
print(f"Images excluded: {excluded_class_df['image_count'].sum()}")

Total classes: 217
Eligible classes: 174
Excluded classes: 43
Images kept: 883
Images excluded: 117


In [3]:
image_records = []

for _, row in eligible_class_df.iterrows():
    class_name = row["class_name"]

    for image_path in row["image_paths"]:
        image_records.append({
            "image_path": image_path,
            "class_name": class_name,
        })

image_df = pd.DataFrame(image_records)

print(f"Total eligible images: {len(image_df)}")
image_df.head()

Total eligible images: 883


,image_path,class_name
0,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet
1,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet
2,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet
3,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet
4,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet


In [4]:
class_names = sorted(image_df["class_name"].unique())

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

idx_to_class = {
    index: class_name
    for class_name, index in class_to_idx.items()
}

image_df["label"] = image_df["class_name"].map(class_to_idx)

print(f"Number of classes: {len(class_names)}")
image_df.head()

Number of classes: 174


,image_path,class_name,label
0,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet,0
1,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet,0
2,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet,0
3,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet,0
4,..\data\raw\parrot_images\Andean_parakeet\0000...,Andean_parakeet,0


In [5]:
image_df[["class_name", "label"]].drop_duplicates().sort_values("label").head(10)

,class_name,label
0,Andean_parakeet,0
5,Antipodes_parakeet,1
10,Austral_parakeet,2
15,Australian_ringneck,3
20,Aztec_parakeet,4
25,Azuero_parakeet,5
30,Bald_parrot,6
35,Barred_parakeet,7
40,Black-billed_amazon,8
45,Black-capped_parakeet,9


In [6]:
train_df, temp_df = train_test_split(
    image_df,
    test_size=0.40,
    stratify=image_df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

print(f"Train images: {len(train_df)}")
print(f"Validation images: {len(val_df)}")
print(f"Test images: {len(test_df)}")

print(f"Train classes: {train_df['label'].nunique()}")
print(f"Validation classes: {val_df['label'].nunique()}")
print(f"Test classes: {test_df['label'].nunique()}")

Train images: 529
Validation images: 177
Test images: 177
Train classes: 174
Validation classes: 174
Test classes: 174


In [7]:
import shutil

def copy_split_to_processed(split_df, split_name, output_dir):
    split_dir = output_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    for _, row in split_df.iterrows():
        source_path = row["image_path"]
        class_name = row["class_name"]

        class_output_dir = split_dir / class_name
        class_output_dir.mkdir(parents=True, exist_ok=True)

        destination_path = class_output_dir / source_path.name

        shutil.copy2(source_path, destination_path)

In [8]:
copy_split_to_processed(train_df, "train", PROCESSED_DATA_DIR)
copy_split_to_processed(val_df, "val", PROCESSED_DATA_DIR)
copy_split_to_processed(test_df, "test", PROCESSED_DATA_DIR)

print("Data split copied to processed folder.")

Data split copied to processed folder.


In [9]:
for split_name in ["train", "val", "test"]:
    split_dir = PROCESSED_DATA_DIR / split_name

    image_count = sum(
        1
        for path in split_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    class_count = sum(
        1
        for path in split_dir.iterdir()
        if path.is_dir()
    )

    print(f"{split_name}: {image_count} images across {class_count} classes")

train: 529 images across 174 classes
val: 177 images across 174 classes
test: 177 images across 174 classes
